# Sähkön tuotannon ja kulutuksen tasapaino Suomessa

## Ladataan kaikki tarvittavat datasetit ja muutetaan hetkittäinen data (Lisää ajankkohta) tunnitaiseksi keskiarvoksi, jotta data on yhteensopivaa vanhemman sekä uudemman datan kanssa

### Kokonaiskulutus

In [69]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

In [75]:
# Lataa muuttujat .env tiedostosta
load_dotenv()

# Lataa API avaimen turvallisesti
API_KEY = os.getenv("API_KEY")

In [79]:
 

# Määritää aikarajan
START_TIME = "2022-01-01T00:00:00Z"
END_TIME   = "2025-02-01T00:00:00Z" 

# reaaliaikaisiin mittauksiin perustuvat datasetit (Päivittyy 3 minuutin välein)
DATASETS = {
    "Kokonaiskulutus": 193,      # Reaalinenaikainen kokonaiskulutus
  "Kokonaistuotanto": 192,       # Reaaliaikainen tuotanto
    "Tuulivoimatuotanto": 181,             # Reaaliaikainen tuulituotanto
    "Ydinvoima": 188,          # Reaaliaikainen ydinvoima
    "nettotuontiJaVienti": 194        # Reaaliaikainen tuonti/vienti
}

# Funktio datasettien lataamiseen ja .csv tiedostojen luontiin
def download_fingrid_data(dataset_name, dataset_id, api_key):
    print(f"--- Ladataan {dataset_name} (ID: {dataset_id}) ---")
    
    url = f"https://data.fingrid.fi/api/datasets/{dataset_id}/data"
    headers = {"x-api-key": api_key}
    
    all_rows = []
    page = 1
    
    while True:
        params = {
            "startTime": START_TIME,
            "endTime": END_TIME,
            "format": "json",
            "pageSize": 20000,
            "page": page
        }
        
        # Tee pyyntö
        try:
            response = requests.get(url, headers=headers, params=params)
        except Exception as e:
            print(f"   ! Yhteysvirhe: {e}")
            break
        
        # Käsittele liian monta pyyntö virhe 429
        if response.status_code == 429:
            print(f"   ! Rajoitus saavutettu sivulla {page}. Odotetaan 60s...")
            time.sleep(60)
            continue
        
        # Käsittele muut virheet
        if response.status_code != 200:
            print(f"   ! Virhe sivulla {page}: {response.status_code}")
            break
            
        # Käsittele data
        data = response.json()
        current_rows = data.get('data', [])
        
        if len(current_rows) == 0:
            print("   > Lataus valmis.")
            break
            
        all_rows.extend(current_rows)
        
        # Tilannepäivitys
        if page % 5 == 0:
            print(f"   Sivu {page}: Kerätty {len(all_rows)} riviä...")
            
        page += 1
        time.sleep(1.0) # Tauko

    # Luo .csv tiedoston jokaiselle eri datasetille, jotta meidän tarvitsee hakea data vain kerran
    if len(all_rows) > 0:
        df = pd.DataFrame(all_rows)
        
        # Muutetaan päivämääräntpythonin datetime muotoon, jotta python ymmärtää nämä päivämääriksi
        if 'startTime' in df.columns:
            df['startTime'] = pd.to_datetime(df['startTime'])
            df = df.set_index('startTime').sort_index()
        
        # Talenna csv
        filename = f"fingrid_{dataset_name}.csv"
        df.to_csv(filename)
        print(f"   [ONNISTUI] Tallennettu {filename} jossa {len(df)} riviä.\n")
    else:
        print("   EPÄONNISTUI dataa ei löytynyt.\n")

# Funktion ajaminen
print(f"Aoitetaan datan lataaminen\n")

for name, id_number in DATASETS.items():
    download_fingrid_data(name, id_number, API_KEY)

print("Kaikki data ladattu")

Aoitetaan datan lataaminen

--- Ladataan Kokonaiskulutus (ID: 193) ---
   Sivu 5: Kerätty 100000 riviä...
   Sivu 10: Kerätty 200000 riviä...
   Sivu 15: Kerätty 300000 riviä...
   Sivu 20: Kerätty 400000 riviä...
   Sivu 25: Kerätty 500000 riviä...
   > Lataus valmis.
   [ONNISTUI] Tallennettu fingrid_Kokonaiskulutus.csv jossa 577165 riviä.

--- Ladataan Kokonaistuotanto (ID: 192) ---
   Sivu 5: Kerätty 100000 riviä...
   Sivu 10: Kerätty 200000 riviä...
   Sivu 15: Kerätty 300000 riviä...
   Sivu 20: Kerätty 400000 riviä...
   ! Virhe sivulla 25: 500
   [ONNISTUI] Tallennettu fingrid_Kokonaistuotanto.csv jossa 480000 riviä.

--- Ladataan Tuulivoimatuotanto (ID: 181) ---
   Sivu 5: Kerätty 100000 riviä...
   Sivu 10: Kerätty 200000 riviä...
   Sivu 15: Kerätty 300000 riviä...
   Sivu 20: Kerätty 400000 riviä...
   Sivu 25: Kerätty 500000 riviä...
   > Lataus valmis.
   [ONNISTUI] Tallennettu fingrid_Tuulivoimatuotanto.csv jossa 574449 riviä.

--- Ladataan Ydinvoima (ID: 188) ---
   Si